# 🏆 Prédiction de Résultats de Matchs de Football

Modèle de machine learning pour prédire les résultats des matchs de football basé sur les statistiques historiques des équipes.

In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn ipywidgets -q

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1️⃣ Installation et Imports des Librairies

In [ ]:
import pandas as pd
import numpy as np

## 2️⃣ Récupération des Données

# Context/lien

In [ ]:
Url_matches = "https://raw.githubusercontent.com/quentinete2/Data_ai_Foot_Quentin_Thibaud/master/Ressources/Data/matches.csv"
Url_tournaments = "https://raw.githubusercontent.com/quentinete2/Data_ai_Foot_Quentin_Thibaud/master/Ressources/Data/tournaments.csv"

df_matches = pd.read_csv(Url_matches)
df_tournaments = pd.read_csv(Url_tournaments)

# Merger les tournaments avec les matches
df_matches = df_matches.merge(df_tournaments[['tournament_id', 'year', 'host_country', 'count_teams']],
                               on='tournament_id', how='left')

print(f"✓ Matches chargés : {len(df_matches)}")
print(f"✓ Tournaments mergés")

✓ Matches chargés : 1248
✓ Tournaments mergés


In [ ]:
# Définir les cible (Match Valide)

conditions = [
    (df_matches['home_team_win'] == 1),
    (df_matches['draw'] == 1),
    (df_matches['away_team_win'] == 1)
]
df_matches['result'] = np.select(conditions, [0, 1, 2], default=np.nan)
df_matches = df_matches.dropna(subset=['result'])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

## 5️⃣ Feature Engineering (Extraction des Caractéristiques)

In [ ]:
# Préparer les données pour le modèle - Feature Engineering
df_matches['match_date'] = pd.to_datetime(df_matches['match_date'])

# Créer des features pour chaque équipe
def create_team_features(df, team_column, score_column):
    """Crée des statistiques par équipe"""
    team_stats = {}
    for team in df[team_column].unique():
        team_data = df[df[team_column] == team]
        team_stats[team] = {
            'avg_goals': team_data[score_column].mean(),
            'total_matches': len(team_data),
            'wins': len(team_data[team_data['result'] == 0]) if 'result' in df.columns else 0,
        }
    return team_stats

# Calculer les stats des équipes à domicile et à l'extérieur
home_stats = create_team_features(df_matches[df_matches['home_team_score'].notna()], 'home_team_name', 'home_team_score')
away_stats = create_team_features(df_matches[df_matches['away_team_score'].notna()], 'away_team_name', 'away_team_score')

print(f"✓ Équipes analysées : {len(home_stats)}")
print(f"✓ Matches totaux : {len(df_matches)}")


✓ Équipes analysées : 84
✓ Matches totaux : 1248


In [ ]:
# Préparer le dataset pour le modèle de prédiction
df_model = df_matches[['home_team_name', 'away_team_name', 'home_team_score', 'away_team_score', 'tournament_name', 'match_date', 'year', 'host_country', 'count_teams']].copy()
df_model = df_model.dropna(subset=['home_team_score', 'away_team_score'])
df_model['home_team_score'] = df_model['home_team_score'].astype(int)
df_model['away_team_score'] = df_model['away_team_score'].astype(int)

# Créer la cible (0=victoire équipe A, 1=match nul, 2=victoire équipe B)
conditions = [
    (df_model['home_team_score'] > df_model['away_team_score']),
    (df_model['home_team_score'] == df_model['away_team_score']),
    (df_model['home_team_score'] < df_model['away_team_score'])
]
df_model['result'] = np.select(conditions, [0, 1, 2])

# Ajouter des features statistiques
def add_team_features_to_df(row, home_stats, away_stats, team_type='home'):
    team_col = 'home_team_name' if team_type == 'home' else 'away_team_name'
    team = row[team_col]
    stats = home_stats if team_type == 'home' else away_stats

    if team in stats:
        return pd.Series({
            f'{team_type}_avg_goals': stats[team]['avg_goals'],
            f'{team_type}_total_matches': stats[team]['total_matches'],
            f'{team_type}_wins': stats[team]['wins']
        })
    return pd.Series({
        f'{team_type}_avg_goals': 0,
        f'{team_type}_total_matches': 0,
        f'{team_type}_wins': 0
    })

home_features = df_model.apply(lambda row: add_team_features_to_df(row, home_stats, away_stats, 'home'), axis=1)
away_features = df_model.apply(lambda row: add_team_features_to_df(row, home_stats, away_stats, 'away'), axis=1)

df_model = pd.concat([df_model, home_features, away_features], axis=1)
df_model['goal_diff'] = df_model['home_avg_goals'] - df_model['away_avg_goals']

# Ajouter des features contextuelles du tournoi
df_model['year'] = df_model['year'].fillna(df_model['year'].median()).astype(int)
df_model['count_teams'] = df_model['count_teams'].fillna(df_model['count_teams'].median()).astype(int)


print(f"✓ Dataset préparé avec {len(df_model)} matches")
print(f"✓ Distribution des résultats : 0={sum(df_model['result']==0)}, 1={sum(df_model['result']==1)}, 2={sum(df_model['result']==2)}")

✓ Dataset préparé avec 1248 matches
✓ Distribution des résultats : 0=677, 1=253, 2=318


## 6️⃣ Entraînement et Validation du Modèle

In [ ]:
# Entraîner le modèle de prédiction
features_for_model = ['home_avg_goals', 'away_avg_goals', 'goal_diff', 'home_total_matches', 'away_total_matches', 'home_wins', 'away_wins', 'year', 'count_teams']

X = df_model[features_for_model].fillna(0)
y = df_model['result']

# Diviser les données
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entraîner le modèle
model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
model.fit(X_train, y_train)

# Évaluer le modèle
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"✓ Modèle entraîné avec succès")
print(f"✓ Précision du modèle : {accuracy:.2%}")
print("\nRapport de classification :")
print(classification_report(y_test, y_pred, target_names=['Victoire Équipe A', 'Match Nul', 'Victoire Équipe B']))


✓ Modèle entraîné avec succès
✓ Précision du modèle : 64.80%

Rapport de classification :
                   precision    recall  f1-score   support

Victoire Équipe A       0.70      0.83      0.76       143
        Match Nul       0.38      0.19      0.25        47
Victoire Équipe B       0.61      0.57      0.59        60

         accuracy                           0.65       250
        macro avg       0.56      0.53      0.53       250
     weighted avg       0.62      0.65      0.62       250



## 7️⃣ Fonction de Prédiction

In [ ]:
# Calculate median year and count_teams from df_model for prediction consistency
# These were used during feature engineering to fill NaNs
median_year = df_model['year'].median()
median_count_teams = df_model['count_teams'].median()

# Fonction pour prédire le résultat d'un match
def predict_match(team_home, team_away):
    """Prédit le résultat d'un match entre deux équipes"""

    # Récupérer les stats
    home_stat = home_stats.get(team_home, {'avg_goals': 0, 'total_matches': 0, 'wins': 0})
    away_stat = away_stats.get(team_away, {'avg_goals': 0, 'total_matches': 0, 'wins': 0})

    # Créer les features
    features = np.array([[
        home_stat['avg_goals'],
        away_stat['avg_goals'],
        home_stat['avg_goals'] - away_stat['avg_goals'],
        home_stat['total_matches'],
        away_stat['total_matches'],
        home_stat['wins'],
        away_stat['wins'],
        median_year,         # Added 'year' feature
        median_count_teams   # Added 'count_teams' feature
    ]])

    # Prédire
    prediction = model.predict(features)[0]
    probabilities = model.predict_proba(features)[0]

    results = {
        'home_team': team_home,
        'away_team': team_away,
        'prediction': ['Victoire ' + team_home, 'Match Nul', 'Victoire ' + team_away][int(prediction)],
        'confidence': probabilities[int(prediction)],
        'probabilities': {
            'home_win': probabilities[0],
            'draw': probabilities[1],
            'away_win': probabilities[2]
        },
        'home_stats': home_stat,
        'away_stats': away_stat
    }

    return results

# Tester la fonction
prediction_result = predict_match('France', 'Mexico')
print(f"\n🎯 PRÉDICTION : {prediction_result['home_team']} vs {prediction_result['away_team']}")
print(f"Résultat prédit : {prediction_result['prediction']}")
print(f"Confiance : {prediction_result['confidence']:.2%}")
print(f"\nProbabilités :")
print(f"  • Victoire {prediction_result['home_team']} : {prediction_result['probabilities']['home_win']:.2%}")
print(f"  • Match Nul : {prediction_result['probabilities']['draw']:.2%}")
print(f"  • Victoire {prediction_result['away_team']} : {prediction_result['probabilities']['away_win']:.2%}")


🎯 PRÉDICTION : France vs Mexico
Résultat prédit : Victoire France
Confiance : 48.29%

Probabilités :
  • Victoire France : 48.29%
  • Match Nul : 31.60%
  • Victoire Mexico : 20.12%


## 9️⃣ Résumé et Statistiques Finales

In [ ]:
# Résumé et statistiques globales
print("\n" + "="*80)
print("📊 RÉSUMÉ DU MODÈLE DE PRÉDICTION")
print("="*80)

# Top 10 équipes par nombre de victoires (filtrées par un minimum de matchs)
top_winners = []
min_matches_for_ranking = 5 # Define a minimum number of matches for ranking

for team, stats in home_stats.items():
    if stats['total_matches'] >= min_matches_for_ranking: # Filter teams with at least min_matches_for_ranking
        win_percentage = (stats['wins'] / stats['total_matches']) * 100 if stats['total_matches'] > 0 else 0
        top_winners.append({'team': team, 'wins': stats['wins'], 'matches': stats['total_matches'], 'win_percentage': win_percentage})

top_winners_df = pd.DataFrame(top_winners).sort_values('win_percentage', ascending=False).head(10)
print(f"\n🏆 Top 10 Équipes (par pourcentage de victoires, min {min_matches_for_ranking} matchs) :")
# Format win_percentage as a percentage string for better readability
top_winners_df['win_percentage'] = top_winners_df['win_percentage'].map('{:.2f}%'.format)
print(top_winners_df.to_string(index=False))

# Équipes avec meilleure moyenne de buts
print("\n\n⚽ Top 10 Équipes (moyenne de buts) :")
top_goals = []
for team, stats in home_stats.items():
    if stats['total_matches'] > 5:  # Au moins 5 matches
        top_goals.append({'team': team, 'avg_goals': stats['avg_goals'], 'matches': stats['total_matches']})

top_goals_df = pd.DataFrame(top_goals).sort_values('avg_goals', ascending=False).head(10)
print(top_goals_df.to_string(index=False))

print("\n\n" + "="*80)
print(f"✓ Modèle prêt à faire des prédictions!")
print(f"✓ Précision globale du modèle : {accuracy:.2%}")
print(f"✓ Total de matches analysés : {len(df_model)}")
print(f"✓ Total d'équipes uniques : {len(home_stats)}")
print("="*80)


📊 RÉSUMÉ DU MODÈLE DE PRÉDICTION

🏆 Top 10 Équipes (par pourcentage de victoires, min 5 matchs) :
          team  wins  matches win_percentage
       Hungary    15       18         83.33%
Czechoslovakia     9       11         81.82%
    Yugoslavia    13       16         81.25%
  West Germany    31       39         79.49%
        Norway    20       26         76.92%
      Portugal    16       21         76.19%
        Brazil    79      106         74.53%
       Austria    10       14         71.43%
       Germany    46       65         70.77%
     Argentina    46       66         69.70%


⚽ Top 10 Équipes (moyenne de buts) :
          team  avg_goals  matches
       Hungary   4.055556       18
        Russia   3.166667        6
        Norway   2.730769       26
    Yugoslavia   2.625000       16
  West Germany   2.461538       39
Czechoslovakia   2.454545       11
      Portugal   2.428571       21
 United States   2.355556       45
       Germany   2.338462       65
       Austria   